# Proyecto de Grado EAFIT — Piloto KKBox en 5 categorías de riesgo

**Daniel Restrepo Ospina** · Seminario de Proyecto de Grado  
**Asesor:** Juan Alejandro Peña Palacio

---

### Qué es este notebook

Este es mi cuaderno de trabajo en **Google Colab**. Aquí dejo el paso a paso de lo que hicimos en la Fase 1:

1. Entender los datos de KKBox (streaming musical, Asia).
2. Tomar una muestra de **1.000 usuarios** (como acordé con mi profesor).
3. Agruparlos en **5 categorías de riesgo de retiro** con K-Means.
4. Ver si esas categorías tienen sentido mirando el **% de churn real**.
5. Exportar un Excel con **5 hojas** (una por categoría).

> **Idea clave:** No quiero solo predecir sí/no churn. Quiero **segmentar en 5 niveles de riesgo**, como hacen aseguradoras o DataCrédito, para decidir **qué hacer con cada grupo** (promo, llamar, o no tocar — *do not poke the bear*).

**Repo:** https://github.com/danielrpo1/pdgrado


---
## Paso 0 — Abrir en Colab

1. Sube este notebook a Colab, o abre desde GitHub: `notebooks/Colab_Piloto_5_Categorias_Riesgo_KKBox.ipynb`
2. Activa GPU si quieres (no es obligatorio para 1.000 usuarios).
3. Ejecuta las celdas en orden (**Runtime → Run all**).


In [ ]:
#@title 0.1 Instalación y clon del repositorio
!pip install -q kaggle pyarrow openpyxl scikit-learn seaborn matplotlib

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/danielrpo1/pdgrado.git
    %cd pdgrado
else:
    # Si corres en local, asume que ya estás en la carpeta del proyecto
    pass

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
print('Directorio de trabajo:', ROOT)
print('¿Colab?:', IN_COLAB)


---
## Paso 1 — Descargar los datos

Los archivos vienen de la competencia **KKBox Churn** (WSDM 2018). En mi máquina los bajé del dataset público `qmdo97/kkboxdataset` en Kaggle (mismos CSV que usa el repo [apostaremczak/churn-prediction](https://github.com/apostaremczak/churn-prediction)).

| Archivo | Qué trae |
|---------|----------|
| `train_v2.csv` | Usuario (`msno`) + si se fue (`is_churn`) |
| `members_v3.csv` | Perfil: ciudad, género, cómo se registró |
| `transactions_v2.csv` | Pagos, auto-renew, cancelaciones, precios |

**No uso** `user_logs` (~30 GB) en este piloto — lo dejamos para una fase después.


In [ ]:
#@title 1.1 Descarga con Kaggle (elige UNA opción)

from pathlib import Path
DATA_RAW = ROOT / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)

# --- OPCIÓN A: Colab + Secrets (recomendada) ---
# En el panel 🔑 Secrets de Colab, crea KAGGLE_API_TOKEN con tu token de
# https://www.kaggle.com/settings → Create New Token → access_token

USE_KAGGLE_DOWNLOAD = True  # pon False si ya subiste los CSV a Drive

if USE_KAGGLE_DOWNLOAD:
    import os
    if 'google.colab' in sys.modules:
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    elif Path.home().joinpath('.kaggle/access_token').exists():
        os.environ['KAGGLE_API_TOKEN'] = Path.home().joinpath('.kaggle/access_token').read_text().strip()

    DATASET = 'qmdo97/kkboxdataset'
    for fname in ['train_v2.csv', 'members_v3.csv', 'transactions_v2.csv']:
        dest = DATA_RAW / fname
        if dest.exists():
            print(f'✓ Ya existe {fname}')
            continue
        print(f'Descargando {fname}...')
        !kaggle datasets download -d {DATASET} -f {fname} -p {DATA_RAW} --force
        !unzip -o -q {DATA_RAW}/{fname}.zip -d {DATA_RAW}
        !(rm -f {DATA_RAW}/{fname}.zip)

# --- OPCIÓN B: Google Drive (si ya tienes los CSV) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_RAW = Path('/content/drive/MyDrive/pdgrado/data/raw')

for f in ['train_v2.csv', 'members_v3.csv', 'transactions_v2.csv']:
    p = DATA_RAW / f
    print(('✓' if p.exists() else '✗'), f, f'({p.stat().st_size/1e6:.1f} MB)' if p.exists() else '')


---
## Paso 2 — ¿Qué problema estoy resolviendo?

En mi trabajo (suscripciones / streaming) el riesgo no es solo “¿se va?”. Me interesa:

- **Categoría 0–1:** usuarios más estables → casi no intervenir.
- **Categoría 2–3:** riesgo medio → ofertas selectivas.
- **Categoría 4:** riesgo alto → acción fuerte o, a veces, **no contactar** para no recordarles la suscripción.

Mi profesor pidió **5 clusters** antes de montar una red neuronal, para que el modelo tenga **interpretabilidad**.


In [ ]:
#@title 2.1 Primer vistazo a los datos

import pandas as pd
import numpy as np

from src.data_io import load_train, load_members, load_transactions

train = load_train(DATA_RAW)
members = load_members(DATA_RAW)
transactions = load_transactions(DATA_RAW)

print('=== Tamaños ===')
print(f'train:         {train.shape[0]:,} usuarios')
print(f'members:       {members.shape[0]:,} perfiles')
print(f'transactions:  {transactions.shape[0]:,} filas de pago')

churn_rate_global = train['is_churn'].mean()
print(f'\nChurn global en train: {churn_rate_global:.1%}')
print('(La base está desbalanceada: pocos se van, muchos renuevan)')

display(train.head(3))
display(transactions.head(3))


### Mi lectura rápida (EDA)

- **`is_churn = 1`**: el usuario **no renovó** dentro de los 30 días después de que venció la membresía.
- **`is_cancel = 1`**: canceló activamente el plan (no siempre es churn definitivo, pero es señal fuerte).
- **`is_auto_renew`**: si tiene cobro automático; en mi experiencia laboral, cuando lo apagan suele ser mala señal.

Variables **numéricas** (pagos, días, montos) → entran al clustering.  
Variables **categóricas** (ciudad, género) → solo para **describir** cada grupo después.


In [ ]:
#@title 2.2 Visual: ¿qué tan desbalanceado está el churn?

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

counts = train['is_churn'].value_counts().sort_index()
ax[0].bar(['Renueva (0)', 'Churn (1)'], counts.values, color=['#2ecc71', '#e74c3c'])
ax[0].set_title('Cuántos usuarios hay por etiqueta')
ax[0].set_ylabel('Cantidad de usuarios')

ax[1].pie(counts.values, labels=['Renueva', 'Churn'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
ax[1].set_title('Proporción en train_v2')

plt.suptitle('Contexto: la clase minoritaria es la que más me interesa', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


---
## Paso 3 — Muestra piloto de 1.000 usuarios

**¿Por qué no uso los ~900k de una vez?**  
Porque mi profesor me pidió validar el método con pocos datos primero. Así depuro variables y clusters sin quemar tiempo ni RAM.

Tomo usuarios que estén en **train y en transactions**, y muestreo **estratificado** por `is_churn` para no perder churners en la muestra.


In [ ]:
#@title 3.1 Construir muestra y variables por usuario

from src.config import PILOT_SAMPLE_SIZE, RANDOM_STATE
from src.data_io import sample_users
from src.features import build_user_features, attach_profile_for_characterization

train_with_tx = train[train['msno'].isin(transactions['msno'].unique())]
pilot_msno = sample_users(train_with_tx, PILOT_SAMPLE_SIZE, RANDOM_STATE)

tx_pilot = transactions[transactions['msno'].isin(pilot_msno)]
mem_pilot = members[members['msno'].isin(pilot_msno)]

print(f'Usuarios en piloto: {pilot_msno.nunique()}')
print(f'Filas de transacciones en piloto: {len(tx_pilot):,}')

features = build_user_features(tx_pilot, members=mem_pilot)
df = attach_profile_for_characterization(features, mem_pilot, train)

from src.config import CLUSTER_FEATURE_COLS
feature_cols = [c for c in CLUSTER_FEATURE_COLS if c in df.columns]
print('\nVariables numéricas para clustering:')
print(feature_cols)
df[['msno','is_churn'] + feature_cols[:5]].head()


### Qué significan mis variables (en llano)

| Variable | En palabras simples |
|----------|---------------------|
| `transaction_count` | Cuántos pagos hizo |
| `cancel_count` | Cuántas veces canceló |
| `auto_renew_ratio` | Qué tan seguido dejó el cobro automático |
| `amount_paid_ntd_sum` | Cuánto dinero pagó en total |
| `days_active_span` | Cuántos días lleva pagando en la ventana |
| `days_until_expire` | Días entre último pago y vencimiento |
| `discount_mean` | Descuento promedio (precio lista − pagado) |


---
## Paso 4 — K-Means con k = 5 (mis 5 categorías de riesgo)

**K-Means** agrupa usuarios que son parecidos en las variables numéricas.  
Luego **ordeno** los grupos del menor al mayor % de churn y les pongo etiqueta **0 = bajo riesgo** … **4 = alto riesgo**.

> Los números 0–4 no los inventa K-Means con significado fijo: yo les doy sentido **después**, mirando el churn de cada grupo.


In [ ]:
#@title 4.1 Entrenar K-Means y etiquetar riesgo

from sklearn.metrics import silhouette_score
from src.clustering import fit_kmeans, order_clusters_by_risk, churn_rate_by_cluster

X = df[feature_cols].fillna(0)
model, scaler, labels = fit_kmeans(X)
df['cluster_raw'] = labels.values

sil = silhouette_score(scaler.transform(X), labels)
print(f'Silhouette (qué tan separados están los clusters): {sil:.3f}')
print('(Entre -1 y 1; más alto = mejor separación, pero en datos reales suele ser modesto)')

risk_map = order_clusters_by_risk(df, cluster_col='cluster_raw')
df['risk_category'] = df['cluster_raw'].map(risk_map)

summary = churn_rate_by_cluster(df, cluster_col='risk_category')
summary['churn_pct'] = (summary['churn_rate'] * 100).round(1)
summary['pct_usuarios'] = (summary['n_users'] / len(df) * 100).round(1)
display(summary)


In [ ]:
#@title 4.2 Gráfico principal: churn por categoría de riesgo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, 5))
ax = axes[0]
bars = ax.bar(summary.index.astype(str), summary['churn_pct'], color=colors, edgecolor='white')
ax.set_xlabel('Categoría de riesgo (0=bajo → 4=alto)')
ax.set_ylabel('% churn observado')
ax.set_title('¿Los clusters capturan riesgo real?')
for b, v in zip(bars, summary['churn_pct']):
    ax.text(b.get_x() + b.get_width()/2, v + 1, f'{v:.0f}%', ha='center', fontsize=10)

ax2 = axes[1]
ax2.bar(summary.index.astype(str), summary['n_users'], color=colors, edgecolor='white')
ax2.set_xlabel('Categoría de riesgo')
ax2.set_ylabel('Número de usuarios')
ax2.set_title('Tamaño de cada grupo en el piloto')

plt.suptitle('Resultado piloto — 1.000 usuarios KKBox', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
## Paso 5 — Interpretación (como lo leo yo)

> **Nota:** Los porcentajes exactos cambian un poco cada vez que corres la muestra (semilla aleatoria 42). La **lógica** debería mantenerse: categorías altas con más churn.

### Categoría 0 — Riesgo bajo
- Suele ser el grupo **más grande**.
- Churn **mucho menor** que el promedio del piloto.
- **Qué haría en la empresa:** mantenerlos, comunicación liviana, no gastar cupones caros.

### Categorías 1 y 2 — Riesgo medio-alto
- Empiezan a verse más cancelaciones y menos auto-renew.
- **Qué haría:** pruebas A/B de ofertas, contenido exclusivo, revisar precio.

### Categorías 3 y 4 — Riesgo muy alto
- Churn cercano o igual al **100%** en el piloto (casi todos se fueron en el mes observado).
- Grupos más **pequeños** → usuarios “extremos”.
- **Qué haría:** escalonar a retención humana **o** aplicar *do not poke* si el contacto empeora (algo que ya vimos en mi trabajo).

### ¿Esto cumple lo que pidió mi profesor?
Sí: tengo **5 segmentos**, puedo exportarlos a Excel y **caracterizarlos** antes de la red neuronal.


In [ ]:
#@title 5.1 Perfil numérico por categoría (heatmap)

profile = df.groupby('risk_category')[feature_cols].mean()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(profile_norm.T, annot=False, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'valor normalizado'})
ax.set_title('Perfil promedio por categoría (más rojo = más alto en ese grupo)')
ax.set_xlabel('Categoría de riesgo')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.show()


In [ ]:
#@title 5.2 Algunas variables clave en caja por categoría

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes,
    ['transaction_count', 'cancel_count', 'auto_renew_ratio'],
    ['Cantidad de pagos', 'Cancelaciones', 'Ratio auto-renew']):
    sns.boxplot(data=df, x='risk_category', y=col, ax=ax, palette='RdYlGn_r')
    ax.set_title(title)
    ax.set_xlabel('Riesgo')
plt.tight_layout()
plt.show()


In [ ]:
#@title 5.3 Perfil cualitativo (ciudad y género) — solo descripción

for cat in sorted(df['risk_category'].unique()):
    sub = df[df['risk_category'] == cat]
    print(f'\n━━━ Riesgo {int(cat)} | n={len(sub)} | churn={sub["is_churn"].mean():.1%} ━━━')
    if 'city' in sub.columns:
        top_city = sub['city'].value_counts().head(3)
        print('Ciudades más frecuentes:', dict(top_city))
    if 'gender' in sub.columns:
        print('Género:', dict(sub['gender'].value_counts()))
    print('Promedios clave:')
    print(sub[['transaction_count','cancel_count','auto_renew_ratio','amount_paid_ntd_sum']].mean().round(2).to_string())


---
## Paso 6 — Exportar Excel (5 hojas)

Esto es lo que le quiero mostrar a mi profesor: **un archivo por categoría** para abrir en Excel y explorar usuarios.


In [ ]:
#@title 6.1 Guardar Excel y descargar en Colab

from src.clustering import export_clusters_to_excel

out_dir = ROOT / 'outputs' / 'clusters'
excel_path = export_clusters_to_excel(df, out_dir)
print('Excel guardado en:', excel_path)

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(excel_path))


---
## Paso 7 — Conclusiones y próximos pasos

### Lo que logré en esta fase
1. Bajé y entendí los datos KKBox (`train`, `members`, `transactions`).
2. Construí variables de comportamiento de pago por usuario.
3. Segmenté en **5 categorías de riesgo** con K-Means.
4. Validé que categorías más altas tienen **más churn observado**.
5. Exporté Excel para revisión con mi asesor.

### Próximos pasos (Fase 2)
- [ ] Completar **vacíos en literatura** (clasificación multidimensional).
- [ ] Escalar muestra (5k → 100k) si los perfiles se mantienen.
- [ ] Entrenar **red neuronal** que prediga categoría + **probabilidades**.
- [ ] Conectar con estrategias de retención en mi trabajo.

### Para el seminario
- Documentos en el repo: `docs/01-contexto.md`, `docs/02-vacios-literatura.md`, `docs/03-preguntas-investigacion.md`

---

*Notebook generado para el proyecto de grado — Universidad EAFIT.*
